# 3D Scalar Wave — Material Interaction

A scalar wave travels from the **left side** of the volume and hits the object in the centre.

- **Blue** = wave compression (pushing outward)
- **Red** = wave rarefaction (pulling inward)
- The **coloured shape** is the object — its material controls how much the wave slows, bends, or gets trapped.

Select a shape and material below, press **▶ Run Simulation**, then use the Play button or timeline scrubber.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# ── Material library ──────────────────────────────────────────────────────────
MATERIALS = {
    'Air (baseline)': {
        'speed': 1.00, 'color': '#aaaaaa',
        'desc': 'No object — watch the unobstructed wave for reference',
    },
    'Bismuth': {
        'speed': 0.55, 'color': '#d4a832',
        'desc': 'Real diamagnetic metal — dense enough to slow and bend waves noticeably',
    },
    'Element 115 — Moscovium': {
        'speed': 0.15, 'color': '#cc44ff',
        'desc': 'Superheavy synthetic element — hypothetical near-trapping of wave energy',
    },
    'Exotic metamaterial': {
        'speed': 0.04, 'color': '#00ffdd',
        'desc': 'Theoretical near-zero wave speed — maximum scattering and interference',
    },
}

# ── Object shape masks ─────────────────────────────────────────────────────────
def make_mask(N, shape):
    """Return a boolean (N,N,N) array marking voxels inside the object."""
    c = N // 2
    idx = np.arange(N)
    X, Y, Z = np.meshgrid(idx, idx, idx, indexing='ij')
    r = int(N * 0.17)

    if shape == 'Sphere':
        return (X-c)**2 + (Y-c)**2 + (Z-c)**2 < r**2

    if shape == 'Torus':
        R_maj, r_tube = r * 1.6, r * 0.55
        # Ring in YZ plane so the hole faces the incoming wave (X direction)
        ryz = np.sqrt((Y-c)**2 + (Z-c)**2)
        return (ryz - R_maj)**2 + (X-c)**2 < r_tube**2

    if shape == 'Pyramid':
        h, base = r * 2.2, r * 1.1
        frac = np.clip(((c + h/2) - Z) / h, 0, 1)
        half = base * frac
        return (
            (np.abs(X-c) < half) & (np.abs(Y-c) < half) &
            (Z >= c - h/2) & (Z <= c + h/2)
        )

    if shape == 'Bismuth Crystal':
        # Rhombohedral lattice: cube sheared along body diagonal
        a, sh = r * 1.25, 0.38
        u = (X - c) + sh * (Z - c)
        v = (Y - c) + sh * (Z - c)
        w = (Z - c)
        return (np.abs(u) < a) & (np.abs(v) < a) & (np.abs(w) < a * 0.65)

    return np.zeros((N, N, N), dtype=bool)

# ── 3-D scalar wave FDTD solver ───────────────────────────────────────────────
def run_fdtd(shape, material, N=48, n_frames=70, spf=4):
    """
    Leapfrog scalar-wave FDTD on an N^3 grid.
    Returns (list_of_field_snapshots, object_boolean_mask).
    """
    dx = 1.0
    dt = dx / (np.sqrt(3) * 1.15)           # 3-D CFL condition (c_max = 1)

    obj_mask = make_mask(N, shape)
    c = np.ones((N, N, N))
    c[obj_mask] = MATERIALS[material]['speed']
    cfl2 = (c * dt / dx) ** 2              # Courant number^2 per cell

    # Absorbing boundary: multiply field by a smooth ramp near edges
    w_pml = 8
    p1d = np.ones(N)
    for i in range(w_pml):
        p1d[i] = p1d[N-1-i] = (i / w_pml) ** 2
    pml = p1d[:, None, None] * p1d[None, :, None] * p1d[None, None, :]

    # Point source: left side, aimed at object centre
    sx, sy, sz = N // 7, N // 2, N // 2
    omega = 2 * np.pi * 0.05               # ~20 grid cells per wavelength

    phi      = np.zeros((N, N, N))
    phi_prev = np.zeros((N, N, N))
    snapshots = []

    for fi in range(n_frames):
        for s in range(spf):
            t = fi * spf + s
            phi[sx, sy, sz] += np.sin(omega * t)

            lap = (
                np.roll(phi, -1, 0) + np.roll(phi,  1, 0) +
                np.roll(phi, -1, 1) + np.roll(phi,  1, 1) +
                np.roll(phi, -1, 2) + np.roll(phi,  1, 2) -
                6.0 * phi
            )
            phi_next = 2*phi - phi_prev + cfl2 * lap
            phi_next *= pml
            phi_prev, phi = phi, phi_next

        snapshots.append(phi.copy())

    return snapshots, obj_mask

# ── Plotly 3-D animated figure ────────────────────────────────────────────────
def make_figure(snapshots, obj_mask, shape, material):
    N  = snapshots[0].shape[0]
    st = 2                                  # stride: render at half resolution
    sl = np.s_[::st, ::st, ::st]

    phi_viz = [f[sl].flatten() for f in snapshots]
    obj_viz = obj_mask[sl].astype(float).flatten()

    Nv = snapshots[0][sl].shape[0]
    g  = np.arange(Nv)
    Xg, Yg, Zg = np.meshgrid(g, g, g, indexing='ij')
    xf, yf, zf = Xg.flatten(), Yg.flatten(), Zg.flatten()

    obj_col = MATERIALS[material]['color']

    # Animation frames update only the wave value array (coordinates stay fixed)
    frames = [
        go.Frame(data=[go.Isosurface(value=v)], traces=[0], name=str(i))
        for i, v in enumerate(phi_viz)
    ]

    fig = go.Figure(
        data=[
            # Trace 0: animated wave field
            go.Isosurface(
                x=xf, y=yf, z=zf, value=phi_viz[0],
                isomin=-0.12, isomax=0.12,
                surface=dict(count=6, fill=0.85),
                colorscale='RdBu',
                opacity=0.20,
                showscale=False,
                caps=dict(x_show=False, y_show=False, z_show=False),
                name='Wave field',
            ),
            # Trace 1: static object surface
            go.Isosurface(
                x=xf, y=yf, z=zf, value=obj_viz,
                isomin=0.5, isomax=1.5,
                surface=dict(count=1),
                colorscale=[[0, obj_col], [1, obj_col]],
                opacity=0.35,
                showscale=False,
                caps=dict(x_show=False, y_show=False, z_show=False),
                name=shape,
            ),
        ],
        frames=frames,
    )

    fig.update_layout(
        title=dict(
            text=f'<b>{shape}  ·  {material}</b>',
            font=dict(color='#aabbff', size=13),
        ),
        scene=dict(
            bgcolor='#030318',
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            camera=dict(eye=dict(x=1.5, y=1.4, z=1.1)),
            aspectmode='cube',
        ),
        paper_bgcolor='#030318',
        font=dict(color='#aabbff'),
        width=760, height=640,
        margin=dict(l=0, r=0, t=42, b=90),
        updatemenus=[dict(
            type='buttons', showactive=False,
            bgcolor='#181830', bordercolor='#334488',
            x=0.15, y=0.02,
            buttons=[
                dict(
                    label='\u25b6 Play', method='animate',
                    args=[None, dict(
                        frame=dict(duration=80, redraw=True),
                        fromcurrent=True,
                        transition=dict(duration=0),
                    )],
                ),
                dict(
                    label='\u23f8 Pause', method='animate',
                    args=[[None], dict(
                        frame=dict(duration=0, redraw=False),
                        mode='immediate',
                    )],
                ),
            ],
        )],
        sliders=[dict(
            active=0,
            currentvalue=dict(prefix='Step: ', font=dict(color='#8899bb', size=11)),
            bgcolor='#181830', bordercolor='#334488',
            font=dict(color='#8899bb', size=10),
            len=0.82, x=0.12,
            steps=[
                dict(
                    method='animate', label='',
                    args=[[str(i)], dict(
                        mode='immediate',
                        frame=dict(duration=80, redraw=True),
                        transition=dict(duration=0),
                    )]
                )
                for i in range(len(snapshots))
            ],
        )],
    )
    return fig

# ── Interactive widget UI ─────────────────────────────────────────────────────
def launch():
    header = widgets.HTML(
        '<h3 style="color:#7788ee;font-family:monospace;margin:6px 0 2px">'
        '\u26a1 3D Scalar Wave \u2014 Material Interaction</h3>'
        '<p style="color:#667788;font-size:12px;margin:0 0 10px">'
        'Wave travels left \u2192 right. '
        'Blue = compression, Red = rarefaction. '
        'The coloured shape is the object.</p>'
    )

    shape_dd = widgets.Dropdown(
        options=['Sphere', 'Torus', 'Pyramid', 'Bismuth Crystal'],
        value='Sphere', description='Shape:',
        style={'description_width': '72px'},
        layout=widgets.Layout(width='260px'),
    )
    mat_dd = widgets.Dropdown(
        options=list(MATERIALS.keys()),
        value='Bismuth', description='Material:',
        style={'description_width': '72px'},
        layout=widgets.Layout(width='380px'),
    )
    mat_desc = widgets.HTML(
        f'<span style="color:#778899;font-size:11px">'
        f'{MATERIALS["Bismuth"]["desc"]}</span>'
    )

    def _update_desc(change):
        mat_desc.value = (
            f'<span style="color:#778899;font-size:11px">'
            f'{MATERIALS[change["new"]]["desc"]}</span>'
        )
    mat_dd.observe(_update_desc, names='value')

    run_btn = widgets.Button(
        description='\u25b6 Run Simulation',
        button_style='primary',
        layout=widgets.Layout(width='180px', height='34px'),
    )
    status = widgets.HTML(
        '<span style="color:#667788">Select shape + material, then Run.</span>'
    )
    out = widgets.Output()

    def _on_run(_):
        run_btn.disabled = True
        status.value = '<span style="color:#ddaa22">Computing\u2026 (~10 s)</span>'
        out.clear_output()
        shape    = shape_dd.value
        material = mat_dd.value
        snapshots, mask = run_fdtd(shape, material)
        fig = make_figure(snapshots, mask, shape, material)
        status.value = (
            '<span style="color:#44cc88">'
            'Done \u2014 press \u25b6 Play or drag the timeline.</span>'
        )
        run_btn.disabled = False
        with out:
            fig.show()

    run_btn.on_click(_on_run)

    display(widgets.VBox([
        header,
        widgets.HBox([shape_dd, mat_dd]),
        mat_desc,
        widgets.HBox([run_btn, status]),
        out,
    ]))

print('Setup complete. Run the next cell to open the visualizer.')

In [ ]:
launch()